In [107]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
from scipy.stats.mstats import winsorize

In [108]:
np.random.seed(42)

# Number of rows
n = 300

# Create data
data = {
    "patient_id": np.arange(1, n+1),
    "age": np.random.randint(18, 80, n),
    "gender": np.random.choice(["Male", "Female"], n),
    "region": np.random.choice(["North", "South", "East", "West"], n),  
    "bmi": np.round(np.random.normal(25, 5, n), 2),  
    "blood_pressure": np.round(np.random.normal(120, 15, n), 2),   
    "cholesterol": np.round(np.random.normal(200, 30, n), 2),
    "glucose": np.round(np.random.normal(100, 20, n), 2),
    "disease_risk": np.random.choice([0, 1], n)
}

df = pd.DataFrame(data)

# -----------------------------
# 🔹 Add Missing Values
# -----------------------------
for col in ["age", "gender", "region", "bmi", "cholesterol", "glucose"]:
    df.loc[df.sample(frac=0.1).index, col] = np.nan   # 10% missing

# -----------------------------
# 🔹 Add Outliers
# -----------------------------
# BMI extreme values
df.loc[df.sample(frac=0.05).index, "bmi"] = df["bmi"] * 2

# Blood pressure extreme
df.loc[df.sample(frac=0.05).index, "blood_pressure"] = df["blood_pressure"] + 80

# Cholesterol extreme
df.loc[df.sample(frac=0.05).index, "cholesterol"] = df["cholesterol"] + 150

# Glucose extreme
df.loc[df.sample(frac=0.05).index, "glucose"] = df["glucose"] + 100

# Save dataset
df.to_csv("patient_data.csv", index=False)

print(df.head(),"\n")
print(df.shape,"\n")
print(df.describe(),"\n")

   patient_id   age  gender region    bmi  blood_pressure  cholesterol  \
0           1  56.0  Female    NaN  34.14          122.42       210.61   
1           2  69.0  Female  North  22.89          131.63       191.36   
2           3   NaN    Male  South  32.22          127.08       215.07   
3           4  32.0  Female   West  15.43          177.05       251.62   
4           5  60.0    Male  South    NaN          135.58       275.47   

   glucose  disease_risk  
0   123.24             0  
1    68.97             0  
2   124.77             0  
3    95.70             1  
4      NaN             0   

(300, 9) 

       patient_id         age         bmi  blood_pressure  cholesterol  \
count  300.000000  270.000000  270.000000      300.000000   270.000000   
mean   150.500000   50.296296   26.133778      125.652700   211.609815   
std     86.746758   18.600705    6.861523       21.614569    45.487289   
min      1.000000   18.000000   11.080000       84.430000   121.700000   
25%     75

Part A: Handling Missing Values

In [109]:
# 1. Identify missing values and provide a summary report (percentage per column).

#Checking missing values 
missing=df.isna().sum()

#percentage of missing value 
print((missing/len(df))*100,"\n")

#2.1 Simple Imputer (Numerical): Replace missing BMI with mean or median.
bmi_fill=SimpleImputer(strategy='mean',missing_values=np.nan)
df['bmi']=bmi_fill.fit_transform(df[['bmi']])

#2.2 Simple Imputer (Categorical): Replace missing Region with the most frequent value.
region_fill=SimpleImputer(strategy='most_frequent',missing_values=np.nan)
df['region']=region_fill.fit_transform(df[['region']]).flatten()

#2.3 Most Frequent Imputation: Replace missing Gender with the most common value.
gender_fill=SimpleImputer(strategy='most_frequent',missing_values=np.nan)
df['gender']=gender_fill.fit_transform(df[['gender']]).flatten()

#2.4 Missing Indicator + Random Sample Imputation: Create binary indicator columns for missingness and use random sampling to impute values.
df['age_missing'] = df['age'].isna().astype(int)
random_values = df['age'].dropna().sample(df['age'].isna().sum(), replace=True)
df.loc[df['age'].isna(), 'age'] = random_values.values

#2.5 KNN Imputer: Apply k-Nearest Neighbors imputation for multivariate imputation.
fill_knn = KNNImputer(n_neighbors=3)
df['glucose'] = fill_knn.fit_transform(df[['glucose']]).flatten()

#2.6 MICE Algorithm: Perform chained equations imputation for multiple variables simultaneously.
fill_mice = IterativeImputer(random_state=0)
df['cholesterol'] = fill_mice.fit_transform(df[['cholesterol']]).flatten()

#After Imputation
print(df.isna().sum())


patient_id         0.0
age               10.0
gender            10.0
region            10.0
bmi               10.0
blood_pressure     0.0
cholesterol       10.0
glucose           10.0
disease_risk       0.0
dtype: float64 

patient_id        0
age               0
gender            0
region            0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
disease_risk      0
age_missing       0
dtype: int64


Part B: Handling Outliers

In [110]:
#3. Detect and remove outliers using:
#3.1 Z-score method: Identify patients with extreme cholesterol or glucose values.

col=["cholesterol","glucose"]

# Calculate Z-score
z_scores = stats.zscore(df[col])

# Find outliers (Z > 3)
outliers = np,abs(z_scores > 3)

# Show rows with outliers
print(outliers,"\n")

#3.2 IQR method: Use interquartile range to detect unusual BMI values.
Q1 = df['bmi'].quantile(0.25)
Q3 = df['bmi'].quantile(0.75)

IQR = Q3 - Q1

# Detect outliers
outliers = (df['bmi'] < (Q1 - 1.5 * IQR)) | (df['bmi'] > (Q3 + 1.5 * IQR))

print(outliers,"\n")

#3.3 Percentile method: Cap values below 1st percentile and above 99th percentile.
lower = df['bmi'].quantile(0.01)
upper = df['bmi'].quantile(0.99)
df['bmi'] = df['bmi'].clip(lower, upper)


(<module 'numpy' from 'c:\\Users\\Admin\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\numpy\\__init__.py'>, array([[False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False]

In [111]:
#4. Apply Winsorization to cap extreme outliers instead of removing them.
# Apply winsorization (1% lower, 1% upper)
df['bmi'] = winsorize(df['bmi'], limits=[0.01, 0.01])

In [115]:
#5. Compare dataset shape and summary before vs after outlier treatment.
print(df.shape)
print(df.describe())

(300, 10)
       patient_id         age         bmi  blood_pressure  cholesterol  \
count  300.000000  300.000000  300.000000      300.000000   300.000000   
mean   150.500000   50.630000   26.031578      125.652700   211.609815   
std     86.746758   18.498118    5.940042       21.614569    43.145013   
min      1.000000   18.000000   14.270000       84.430000   121.700000   
25%     75.750000   36.750000   22.240000      111.625000   186.100000   
50%    150.500000   52.000000   26.133778      122.195000   211.435000   
75%    225.250000   66.000000   28.825000      134.527500   229.502500   
max    300.000000   79.000000   48.020000      217.270000   414.910000   

          glucose  disease_risk  age_missing  
count  300.000000    300.000000   300.000000  
mean   105.912593      0.546667     0.100000  
std     28.036698      0.498649     0.300501  
min     32.970000      0.000000     0.000000  
25%     91.700000      0.000000     0.000000  
50%    105.912593      1.000000     0.000

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4842: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4842: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4842: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4842: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4842: UserWarning: Warning: 'partition' will ignore the 'mask' of the 

Part C: Final Clean Dataset

In [118]:
#6 Present the final cleaned dataset with:
#• All missing values treated appropriately.
#• All outliers handled using suitable methods.

#Checking missing values after imputation
print(df.isna().sum(),"\n")

#First 10 rows of the cleaned dataset
print(df.head(),"\n")

#New Dataset
df.to_csv("new_patient_data.csv", index=False)

patient_id        0
age               0
gender            0
region            0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
disease_risk      0
age_missing       0
dtype: int64 

   patient_id   age  gender region        bmi  blood_pressure  cholesterol  \
0           1  56.0  Female  North  34.140000          122.42       210.61   
1           2  69.0  Female  North  22.890000          131.63       191.36   
2           3  69.0    Male  South  32.220000          127.08       215.07   
3           4  32.0  Female   West  15.430000          177.05       251.62   
4           5  60.0    Male  South  26.133778          135.58       275.47   

      glucose  disease_risk  age_missing  
0  123.240000             0            0  
1   68.970000             0            0  
2  124.770000             0            1  
3   95.700000             1            0  
4  105.912593             0            0   



📌 Imputation Strategy Effectiveness

Among all methods used:

- Simple Imputer (Mean/Most Frequent) was effective for:
- BMI (numerical)
- Region (categorical)
- Easy and fast, but slightly reduces data variability
- Random Sample Imputation + Indicator preserved distribution better
- Maintains original data pattern
- Keeps information about missingness
- KNN Imputer was more accurate than simple methods
- Uses similarity between rows
- Better for numerical relationships
- MICE (Iterative Imputer) was the most effective overall
- Predicts values using multiple features
- Produces realistic and statistically strong imputations

👉 Conclusion:
MICE was the most effective, followed by KNN, while Simple Imputer is best for quick/basic filling